In [1]:
import torch

In [2]:
import torch.nn as nn
import torch.functional as f

In [3]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [5]:
batch_size = 32
data_transforms = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 479kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.1MB/s]


In [6]:
class Autoencoders(nn.Module):

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, C, H, W)
        x = x.reshape(x.size(0), -1)

        x = self.encoder(x)
        x = self.decoder(x)

        return x

In [7]:
loss = nn.MSELoss()
model = Autoencoders(input_dim=784, hidden_dim=256, latent_dim=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [8]:
num_epoch = 10

In [9]:
num_epochs = 10

for epoch in range(num_epochs):
    for batch_idx, (data, target) in enumerate(train_loader):

        data = data.to(device)

        # Flatten MNIST image
        data_flat = data.reshape(data.size(0), -1)

        optimizer.zero_grad()

        # Autoencoder reconstructs the input
        y_pred = model(data)

        # Compare reconstruction with original input
        loss_ = loss(y_pred, data_flat)

        loss_.backward()
        optimizer.step()

        if batch_idx % 2 == 0:
            print(
                f"Epoch: {epoch} | "
                f"batch_idx: {batch_idx} | "
                f"loss: {loss_.item():.4f}"
            )

Streaming output truncated to the last 5000 lines.
Epoch: 4 | batch_idx: 1256 | loss: 0.0409
Epoch: 4 | batch_idx: 1258 | loss: 0.0362
Epoch: 4 | batch_idx: 1260 | loss: 0.0444
Epoch: 4 | batch_idx: 1262 | loss: 0.0456
Epoch: 4 | batch_idx: 1264 | loss: 0.0436
Epoch: 4 | batch_idx: 1266 | loss: 0.0442
Epoch: 4 | batch_idx: 1268 | loss: 0.0466
Epoch: 4 | batch_idx: 1270 | loss: 0.0470
Epoch: 4 | batch_idx: 1272 | loss: 0.0511
Epoch: 4 | batch_idx: 1274 | loss: 0.0475
Epoch: 4 | batch_idx: 1276 | loss: 0.0426
Epoch: 4 | batch_idx: 1278 | loss: 0.0450
Epoch: 4 | batch_idx: 1280 | loss: 0.0460
Epoch: 4 | batch_idx: 1282 | loss: 0.0482
Epoch: 4 | batch_idx: 1284 | loss: 0.0493
Epoch: 4 | batch_idx: 1286 | loss: 0.0471
Epoch: 4 | batch_idx: 1288 | loss: 0.0441
Epoch: 4 | batch_idx: 1290 | loss: 0.0489
Epoch: 4 | batch_idx: 1292 | loss: 0.0439
Epoch: 4 | batch_idx: 1294 | loss: 0.0487
Epoch: 4 | batch_idx: 1296 | loss: 0.0460
Epoch: 4 | batch_idx: 1298 | loss: 0.0432
Epoch: 4 | batch_idx: 130

In [10]:
val_dataset = datasets.MNIST(root='./val', train=True, download=True, transform=data_transforms)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 474kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.44MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.0MB/s]


In [9]:
model.eval()

with torch.no_grad():
    for batch_idx, (data, target) in enumerate(val_loader):
        data = data.to(device)